# 🌐 Multi-Site Manga Scraper & PDF Generator for Google Colab
Jalankan notebook ini di **Google Colab** untuk mengunduh komik dari **Komikcast v3** & **Shinigami Scans** secara otomatis dan menyimpannya langsung ke **Google Drive** Anda dalam format **PDF** atau **Folder Gambar**.

---

In [ ]:
#@title 🚀 Step 1: Install Dependencies & Mount Google Drive
!pip install -q requests curl_cffi pillow tqdm

import os
from google.colab import drive

Mount_Google_Drive = True #@param {type:"boolean"}

if Mount_Google_Drive:
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/Manga_Downloads'
    print(f"📁 Output directory: Google Drive ({OUTPUT_DIR})")
else:
    OUTPUT_DIR = '/content/Manga_Downloads'
    print(f"📁 Output directory: Colab Local ({OUTPUT_DIR})")

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
#@title ⚙️ Step 2: Load Multi-Site Scraper Core Engines
import os
import re
import sys
import shutil
import requests
from PIL import Image
from curl_cffi import requests as curl_requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def sanitize_folder_name(name):
    return re.sub(r'[\\/*?:"<>|]', "", str(name)).strip()

def convert_folder_to_pdf(chapter_dir, pdf_save_path):
    valid_exts = ('.jpg', '.png', '.jpeg', '.webp', '.bmp')
    image_files = [
        os.path.join(chapter_dir, f) for f in os.listdir(chapter_dir)
        if f.lower().endswith(valid_exts)
    ]
    image_files.sort()
    if not image_files:
        return False
    pil_images = []
    for filepath in image_files:
        try:
            img = Image.open(filepath)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            pil_images.append(img)
        except Exception:
            pass
    if pil_images:
        os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
        pil_images[0].save(pdf_save_path, save_all=True, append_images=pil_images[1:])
        print(f"📄 PDF Created: {pdf_save_path}")
        return True
    return False

# -------------------- KOMIKCAST --------------------
class KomikcastEngine:
    BASE_API = "https://be.komikcast.cc"
    HEADERS = {'User-Agent': 'Mozilla/5.0', 'Origin': 'https://v3.komikcast.fit', 'Referer': 'https://v3.komikcast.fit/'}
    
    @staticmethod
    def extract_slug(input_str):
        match = re.search(r'series/([^/]+)', input_str)
        return match.group(1) if match else input_str.strip().rstrip('/')
        
    def get_info(self, url_or_slug):
        slug = self.extract_slug(url_or_slug)
        res = requests.get(f"{self.BASE_API}/series/{slug}", headers=self.HEADERS, timeout=10)
        data = res.json().get("data", {})
        inner = data.get("data", {})
        return {"title": inner.get("title", slug), "slug": inner.get("slug", slug), "author": inner.get("author", "Unknown")}
        
    def get_chapters(self, slug):
        res = requests.get(f"{self.BASE_API}/series/{slug}/chapters", headers=self.HEADERS, timeout=10)
        chapters = res.json().get("data", [])
        return sorted(chapters, key=lambda x: float(x.get("data", {}).get("index") or x.get("index") or 0))
        
    def get_details(self, slug, chapter_item):
        ch_idx = chapter_item.get("data", {}).get("index") or chapter_item.get("index")
        res = requests.get(f"{self.BASE_API}/series/{slug}/chapters/{ch_idx}", headers=self.HEADERS, timeout=10)
        data = res.json().get("data", {})
        inner = data.get("data", {})
        return {"number": ch_idx, "title": inner.get("title"), "images": inner.get("images", [])}

# -------------------- SHINIGAMI --------------------
class ShinigamiEngine:
    BASE_API = "https://api.shngm.io/v1"
    HEADERS = {'User-Agent': 'Mozilla/5.0', 'Origin': 'https://g.shinigami.asia', 'Referer': 'https://g.shinigami.asia/'}
    
    def __init__(self):
        self.session = curl_requests.Session(impersonate="chrome120")
        
    @staticmethod
    def extract_id(input_str):
        match = re.search(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', input_str, re.IGNORECASE)
        return match.group(0) if match else input_str.strip().rstrip('/')
        
    def get_info(self, url_or_id):
        manga_id = self.extract_id(url_or_id)
        res = self.session.get(f"{self.BASE_API}/manga/detail/{manga_id}", headers=self.HEADERS, timeout=10)
        data = res.json().get("data", {})
        return {"title": data.get("title", manga_id), "slug": manga_id, "author": data.get("author_name") or "Unknown"}
        
    def get_chapters(self, manga_id):
        manga_id = self.extract_id(manga_id)
        all_ch = []
        page = 1
        while True:
            res = self.session.get(f"{self.BASE_API}/chapter/{manga_id}/list?page={page}&pageSize=100", headers=self.HEADERS, timeout=10)
            data = res.json()
            ch = data.get("data", [])
            if not ch: break
            all_ch.extend(ch)
            meta = data.get("meta", {})
            if page >= (meta.get("total_page") or 1): break
            page += 1
        return sorted(all_ch, key=lambda x: float(x.get("chapter_number") or 0))
        
    def get_details(self, slug, chapter_item):
        ch_id = chapter_item.get("chapter_id")
        res = self.session.get(f"{self.BASE_API}/chapter/detail/{ch_id}", headers=self.HEADERS, timeout=10)
        data = res.json().get("data", {})
        base_url = data.get("base_url", "https://assets.shngm.id").rstrip('/')
        info = data.get("chapter", {})
        path = info.get("path", "").strip('/')
        fnames = info.get("data", [])
        urls = [f"{base_url}/{path}/{f}" if path else f"{base_url}/{f}" for f in fnames]
        return {"number": data.get("chapter_number"), "title": data.get("chapter_title", ""), "images": urls}

print("✅ Multi-site Scraper Engines Ready!")


In [ ]:
#@title 📥 Step 3: Run Interactive Downloader Form

Website = "Auto-Detect" #@param ["Auto-Detect", "Komikcast", "Shinigami Scans"]
URL_atau_Identifier = "https://g.shinigami.asia/series/fb9be087-b8b4-4a26-b088-58fbc7cfce49" #@param {type:"string"}
Mode_Download = "Chapter Terbaru Saja (Test)" #@param ["Chapter Terbaru Saja (Test)", "Semua Chapter", "Range Chapter (Custom)"]
Chapter_Awal = 1 #@param {type:"number"}
Chapter_Akhir = 10 #@param {type:"number"}
Format_Output = "PDF + Gambar" #@param ["PDF + Gambar", "PDF Saja", "Gambar Saja"]

url_lower = URL_atau_Identifier.lower()
if Website == "Komikcast" or ("komikcast" in url_lower and Website == "Auto-Detect"):
    engine = KomikcastEngine()
    site_name = "Komikcast"
else:
    engine = ShinigamiEngine()
    site_name = "Shinigami Scans"

print(f"🌐 Engine terdeteksi: [{site_name}]")
make_pdf = Format_Output in ["PDF + Gambar", "PDF Saja"]
keep_images = Format_Output in ["PDF + Gambar", "Gambar Saja"]

try:
    info = engine.get_info(URL_atau_Identifier)
    print(f"📖 Series Title: {info['title']}")
    print(f"✍️ Author: {info['author']}")
    
    chapters = engine.get_chapters(info['slug'])
    print(f"📚 Total Chapters Found: {len(chapters)}")
    
    def download_ch(ch_item):
        dt = engine.get_details(info['slug'], ch_item)
        imgs = dt.get("images", [])
        if not imgs: return
        ch_name = f"Chapter {dt['number']}"
        ch_dir = os.path.join(OUTPUT_DIR, sanitize_folder_name(info['title']), ch_name)
        os.makedirs(ch_dir, exist_ok=True)
        
        def dl_img(u, p):
            if os.path.exists(p) and os.path.getsize(p) > 0: return
            req = engine.session.get(u) if hasattr(engine, 'session') else requests.get(u)
            if req.status_code == 200:
                with open(p, "wb") as f: f.write(req.content)
                
        with ThreadPoolExecutor(max_workers=8) as ex:
            futures = []
            for i, u in enumerate(imgs, 1):
                ext = u.split('.')[-1].split('?')[0]
                if len(ext) > 4: ext = "webp"
                futures.append(ex.submit(dl_img, u, os.path.join(ch_dir, f"{i:03d}.{ext}")))
            for _ in tqdm(as_completed(futures), total=len(futures), desc=f"Ch {dt['number']}", leave=False):
                pass
                
        if make_pdf:
            pdf_path = os.path.join(OUTPUT_DIR, sanitize_folder_name(info['title']), f"{ch_name}.pdf")
            convert_folder_to_pdf(ch_dir, pdf_path)
            if not keep_images:
                shutil.rmtree(ch_dir, ignore_errors=True)

    if Mode_Download == "Semua Chapter":
        print(f"🚀 Downloading ALL {len(chapters)} chapters...")
        for ch in tqdm(chapters, desc="Total Progress"):
            download_ch(ch)
    elif Mode_Download == "Range Chapter (Custom)":
        filtered = [c for c in chapters if Chapter_Awal <= float(engine.get_details(info['slug'], c)['number'] or 0) <= Chapter_Akhir]
        print(f"🎯 Downloading Range Chapter {Chapter_Awal} - {Chapter_Akhir} ({len(filtered)} chapters)...")
        for ch in tqdm(filtered, desc="Range Progress"):
            download_ch(ch)
    else:
        latest = chapters[-1]
        print(f"🧪 Downloading Latest Chapter...")
        download_ch(latest)
        
    print(f"\n🎉 DONE! Files stored in: {os.path.join(OUTPUT_DIR, sanitize_folder_name(info['title']))}")

except Exception as e:
    print(f"❌ Error: {e}")
